# Which words split Phi-3-mini's attention graph

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vcnoel/spectral-trust/blob/main/notebooks/04_phi3_prepositions.ipynb)

In Phi-3-mini-4k-instruct, the Fiedler value λ₂ of the head-averaged attention
graph at layers 3 to 5 is close to 0.93 for most short sentences and falls to
about 0.37 for some. Passive sentences are among those that fall, so the drop
looks like a passive-voice effect. It is not: the
[use-case page](https://vcnoel.github.io/spectral-trust/use-cases/phi3-prepositions/)
traces it to a set of frequent prepositions. This notebook reruns three of its
measurements: the trigger-word probe (to, of, by against near, behind,
after), λ₂ against attention-sink mass on 72 active/passive pairs, and the
Fiedler split of one sentence.

**Runtime:** a T4 GPU (Colab: *Runtime > Change runtime type > T4 GPU*).
Phi-3-mini runs in float16, which needs about 8 GB of GPU memory; the first
run downloads about 7.6 GB. The measurements then take a few minutes. The
use-case page ran Phi-3 in bfloat16, which the T4 does not support natively,
so values here can differ from the page's in the third decimal.

*Tested on a GPU runtime: not yet.* The code path was run end to end on CPU
with a small model (Qwen3-0.6B) in place of Phi-3-mini.

In [ ]:
%pip install -q "spectral-trust[viz]==0.5.0"

In [ ]:
import os
os.environ.setdefault("TQDM_DISABLE", "1")          # no progress bars in the outputs

import numpy as np
import torch
import transformers

import spectral_trust as st

transformers.logging.set_verbosity_error()
transformers.utils.logging.disable_progress_bar()
print("spectral-trust", st.__version__, "| torch", torch.__version__,
      "| transformers", transformers.__version__)

## Load Phi-3-mini

Phi-3's tokenizer adds no BOS token, so position 0, the attention sink, is
the first word of the sentence.

In [ ]:
MODEL = "microsoft/Phi-3-mini-4k-instruct"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = "float16" if DEVICE == "cuda" else "float32"
if DEVICE == "cpu" and "Phi-3" in MODEL:
    raise RuntimeError("Phi-3-mini needs a GPU runtime here: "
                       "Runtime > Change runtime type > T4 GPU, then run all cells again.")

model = st.SpectralModel.from_pretrained(MODEL, dtype=DTYPE, device_map=DEVICE)
model.model.eval()
print(model, DEVICE, DTYPE)


def measure(texts, layers):
    """λ2 of the head-averaged graph and sink mass (attention received by
    position 0, mean over heads and over queries 1..), [texts, layers]."""
    cache = model.run_with_spectra(texts, metrics=["graph.fiedler_value"], layers=layers,
                                   hybrid=False, batch_size=1, keep_patterns=True)
    lam2 = np.stack([np.asarray(cache[f"blocks.{l}.graph.fiedler_value"], float) for l in layers], 1)
    sink = np.array([[np.asarray(cache[i][f"blocks.{l}.attn.pattern"], float).mean(0)[1:, 0].mean()
                      for l in layers] for i in range(len(texts))])
    return lam2, sink


lam2, _ = measure(["The scientist discovered the particle.",
                   "The particle was discovered by the scientist."], [3])
print("λ2 at layer 3: active", round(lam2[0, 0], 3), "| passive", round(lam2[1, 0], 3))

## The trigger-word probe

Actives of the form "The cat slept by the door.", with six prepositions in
eight frames each. If the drop were about the passive voice, none of these
sentences would drop.

In [ ]:
import pandas as pd

PREPS = ["to", "of", "by", "near", "behind", "after"]
FRAMES = [("cat", "slept", "door"), ("man", "waited", "window"), ("dog", "sat", "fire"),
          ("woman", "stood", "gate"), ("boat", "drifted", "shore"), ("girl", "read", "lamp"),
          ("soldier", "rested", "wall"), ("bird", "landed", "pond")]
texts = [f"The {s} {v} {p} the {o}." for p in PREPS for s, v, o in FRAMES]
words = [p for p in PREPS for _ in FRAMES]
lam2, sink = measure(texts, [3])
probe = pd.DataFrame({"word": words, "λ2 at layer 3": lam2[:, 0], "sink mass at layer 3": sink[:, 0]})
probe.groupby("word", sort=False).agg(["mean", "min", "max"]).round(3)

On the use-case page (bfloat16), *to*, *of* and *by* give λ₂ of 0.370 to 0.379
with sink mass 0.669 to 0.679, and *near*, *behind* and *after* 0.928 to
0.931 with sink mass 0.903 to 0.911; the page probes 12 words in all.

## λ₂ against sink mass on 72 active/passive pairs

The pairs come from `examples/use_cases/data/passive_active.json` in the
repository, at the commit of the 0.5.0 release. The two members of a pair
start with the same word, so the sink token is the same. AUROC is that of the
passive value over the active one: 0 means that every passive is below every
active.

In [ ]:
import json
import urllib.request

from scipy.stats import rankdata, spearmanr

DATA = ("https://raw.githubusercontent.com/vcnoel/spectral-trust/"
        "33d6bf8ff0e507368a7ab4bf83bfa0624166cd98/examples/use_cases/data/passive_active.json")
items = [it for it in json.load(urllib.request.urlopen(DATA)) if it["group"] == "en_pair"]
by_pair = {}
for it in items:
    by_pair.setdefault(it["pair"], {})[it["voice"]] = it["text"]
actives = [v["active"] for v in by_pair.values()]
passives = [v["passive"] for v in by_pair.values()]
print(len(actives), "pairs, for example:", actives[0], "/", passives[0])

LAYERS = [2, 3, 4, 5]
lam_a, sink_a = measure(actives, LAYERS)
lam_p, sink_p = measure(passives, LAYERS)


def auroc(neg, pos):
    """P(pos > neg) + 0.5 P(tie)."""
    r = rankdata(np.concatenate([neg, pos]))
    n0, n1 = len(neg), len(pos)
    return (r[n0:].sum() - n1 * (n1 + 1) / 2) / (n0 * n1)


table = pd.DataFrame({
    "layer": LAYERS,
    "active λ2": lam_a.mean(0), "passive λ2": lam_p.mean(0),
    "AUROC λ2": [auroc(lam_a[:, j], lam_p[:, j]) for j in range(len(LAYERS))],
    "AUROC sink mass": [auroc(sink_a[:, j], sink_p[:, j]) for j in range(len(LAYERS))],
    "Spearman λ2, sink (passives)": [spearmanr(lam_p[:, j], sink_p[:, j])[0] for j in range(len(LAYERS))],
}).set_index("layer")
table.round(3)

In [ ]:
import matplotlib.pyplot as plt

j = LAYERS.index(3)
fig, ax = plt.subplots(figsize=(4.8, 3.4))
ax.scatter(sink_a[:, j], lam_a[:, j], s=14, color="#0072B2", label="active")
ax.scatter(sink_p[:, j], lam_p[:, j], s=14, color="#E69F00", label="passive")
ax.set(xlabel="sink mass at layer 3", ylabel="λ2 at layer 3")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

On the page, sink mass separates every pair at layers 3 to 5 as well as λ₂
does, so a spectral drop should be compared with sink mass before it is read
as structure. What λ₂ adds is the location of the split.

## The Fiedler split of one sentence

The Fiedler vector of the head-averaged graph places the tokens along its
weakest cut. Its smaller side, by sign, is the group of tokens that the graph
holds apart:

In [ ]:
text = "The particle was discovered by the scientist."
one = model.run_with_spectra(text, layers=[3], hybrid=False, keep_patterns=True)
u = np.asarray(one["blocks.3.graph.fiedler_vector"])
pos = u > 0
small = pos if (2 * pos.sum() < len(u) or (2 * pos.sum() == len(u) and not pos[0])) else ~pos
print(f"λ2 at layer 3: {one['blocks.3.graph.fiedler_value']:.3f}")
print("tokens:     ", one.tokens)
print("smaller side:", [t for t, k in zip(one.tokens, small) if k])
one.graph(3)

On the page, the smaller side at layer 3 is "the scient ist ." with λ₂ 0.375:
the tokens after the trigger *by* leave the sink together. The page also
measures length-adjusted differences, agentless passives, by-actives, French
and German pairs, four other models, and three edits at layer 2 that do not
undo the split; the scripts are in
[`examples/use_cases/`](https://github.com/vcnoel/spectral-trust/tree/main/examples/use_cases).